In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# 1. Install libraries
!pip install librosa numpy

import os
import numpy as np
import librosa
from google.colab import drive

# ================= Configuration Area =================
# Point to the folder in your Google Drive where the original WAV files are stored
folder_path = "/content/gdrive/MyDrive/music"

# Output filename (After generation, please put this file into the Arduino Nano project folder)
target_filename = "sound_data.h"

# Your list of files
source_files = ["void.wav", "bio.wav", "tech.wav", "material.wav"]

# Target sample rate (16000Hz is the best balance point for Nano mixing)
TARGET_SR = 16000
# ====================================================

def generate_mixing_ready_header():
    print(f">>> Starting to prepare mixing data for Nano RP2040...")
    print(f"    Architecture: Dual-board collaboration (ESP32 Control -> Nano Mixing)")
    print(f"    Target format: Raw PCM (Headerless), 16kHz, 16-bit, Mono")

    header_path = os.path.join(folder_path, target_filename)

    with open(header_path, "w", encoding="utf-8") as out:
        out.write(f"// Generated for Arduino Nano RP2040 Connect (Mixing Node)\n")
        out.write(f"// Raw PCM Data (No WAV Headers) - Ready for mixing math\n")
        out.write("#ifndef SOUND_DATA_H\n#define SOUND_DATA_H\n\n")
        out.write("#include <Arduino.h>\n\n")

        total_bytes = 0

        for fname in source_files:
            full_path = os.path.join(folder_path, fname)

            if not os.path.exists(full_path):
                print(f"[Error] File not found: {full_path}")
                continue

            try:
                # --- Core Optimization Steps ---
                # 1. Load with librosa, automatically convert to mono and resample (sr)
                #    y is a float array, ranging from -1.0 to 1.0
                y, sr = librosa.load(full_path, sr=TARGET_SR, mono=True)

                # 2. Convert to 16-bit integer (This is the raw value needed for mixing)
                #    Map -1.0~1.0 to -32768~32767
                y_int16 = (y * 32767).astype(np.int16)

                # 3. Convert to byte stream (Endianness is usually handled by the compiler, converting to pure bytes here)
                raw_data = y_int16.tobytes()

                # --- Write to .h file ---
                var_name = fname.replace(".", "_") # void.wav -> void_wav

                out.write(f"// Source: {fname} | Sample Rate: {TARGET_SR}Hz | Length: {len(y_int16)} samples\n")
                # Use const to ensure data is stored in Flash (PROGMEM)
                out.write(f"const unsigned char {var_name}[] = {{\n")

                # Write in hexadecimal format
                for i, byte in enumerate(raw_data):
                    out.write(f"0x{byte:02x},")
                    if (i + 1) % 16 == 0:
                        out.write("\n")

                out.write("\n};\n")
                # Record byte length
                out.write(f"const unsigned int {var_name}_len = {len(raw_data)};\n")
                # Additionally record the number of samples (Very useful for the mixing loop)
                out.write(f"const unsigned int {var_name}_samples = {len(y_int16)};\n\n")

                print(f"[Success] {fname}: Conversion complete. Contains {len(y_int16)} samples (Raw PCM)")
                total_bytes += len(raw_data)

            except Exception as e:
                print(f"[Failed] Error processing {fname}: {e}")

        out.write("#endif\n")

    print("-" * 30)
    print(f"Processing complete! Total data size: {total_bytes/1024:.2f} KB")
    print(f"The generated .h file has been saved to Google Drive: {target_filename}")
    print(f"Please ensure this file is flashed to the **Nano RP2040**, not the ESP32.")

# Run
generate_mixing_ready_header()

>>> 开始为 Nano RP2040 准备混音数据...
    架构: 双板协作 (ESP32 控制 -> Nano 混音)
    目标格式: Raw PCM (无头), 16kHz, 16-bit, 单声道
[成功] void.wav: 转换完毕。包含 110126 个样本 (Raw PCM)
[成功] bio.wav: 转换完毕。包含 218417 个样本 (Raw PCM)
[成功] tech.wav: 转换完毕。包含 200743 个样本 (Raw PCM)
[成功] material.wav: 转换完毕。包含 76544 个样本 (Raw PCM)
------------------------------
处理完成！总数据量: 1183.26 KB
生成的 .h 文件已保存至 Google Drive: sound_data.h
请确保将此文件烧录到 **Nano RP2040** 中，而不是 ESP32。
